# ASMBI replication notebook — FINAL, Colab-friendly

**Manuscript:** *Local Topological Organization of Semantic Narrative Spaces: A Distance-Sensitive Statistical Framework for Human and AI-Generated Narratives*

This notebook reproduces the revised core analyses and the manuscript figures, excluding obsolete exploratory analyses.

### Reproducibility notes
- Input: `extracted_corpus_from_orange.csv`.
- Original narrative text is embedded without lowercasing, stopword removal, stemming, lemmatization, or punctuation stripping.
- Embedding model: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` (768 dimensions).
- AI generation model: OpenAI `gpt-3.5-turbo-0301`.
- The original generation design targeted three independent AI generations per Human narrative; after exclusion of empty or unavailable texts, the final analytical corpus contains 183 AI narratives, and 61 prompts retain the complete 1-Human + 3-AI structure.
- The original generation prompt template and the values of temperature, top-p, and maximum-token settings were not retained and cannot be reconstructed reliably.
- Four typographical sample-ID corrections are applied explicitly.
- Primary prompt-aware graph sample: 61 complete blocks = 61 Human + 183 AI narratives.
- Random seeds and replication counts are fixed.
- Outputs are saved in `/content/asmbi_outputs/`.
- The 500-replication stability analysis is optimized to avoid repeated full Mahalanobis distance matrices.

- Manuscript figures use the exact visualization specifications recovered from the original working notebook; revised inferential analyses remain separate where appropriate.


In [ ]:
!pip install -q sentence-transformers scikit-learn scipy pandas numpy networkx ripser persim tqdm umap-learn matplotlib


In [ ]:
from pathlib import Path
import re, warnings, time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import umap
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.covariance import LedoitWolf
from ripser import ripser
from persim import plot_diagrams

DATA_PATH = Path("/content/extracted_corpus_from_orange.csv")
OUTPUT_DIR = Path("/content/asmbi_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
GEN_MODEL = "gpt-3.5-turbo-0301"
PRIMARY_K = 10
SEED_MAIN = 20260924
SEED_BALANCED = 12345
SEED_PH = 12345


## 1. Corpus and prompt structure

In [ ]:
df = pd.read_csv(DATA_PATH)
required = {"sample_id", "label", "text"}
assert required.issubset(df.columns)

df["text"] = df["text"].fillna("").astype(str)
df["label"] = df["label"].astype(str).str.strip()
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

id_corrections = {
    "Samplle_22": "Sample_22",
    "Samplle_26": "Sample_26",
    "Sampe_51": "Sample_51",
    "Sampe_57": "Sample_57",
}
df["sample_id"] = df["sample_id"].replace(id_corrections)

def extract_prompt_id(x):
    m = re.search(r"sample_(\d+)", str(x), flags=re.I)
    return int(m.group(1)) if m else np.nan

df["prompt_id"] = df["sample_id"].apply(extract_prompt_id)
assert df["prompt_id"].notna().all()
df["prompt_id"] = df["prompt_id"].astype(int)

print(df.shape)
print(df["label"].value_counts())
assert len(df) == 247
assert (df["label"]=="Human").sum() == 64
assert (df["label"]=="AI").sum() == 183

## 2. Sentence-transformer embeddings

In [ ]:
model = SentenceTransformer(MODEL_NAME)
embeddings = np.asarray(model.encode(
    df["text"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=False
))
print("Embedding shape:", embeddings.shape)
assert embeddings.shape == (247, 768)

## 3. Distance diagnostics: Euclidean, Manhattan, cosine, and regularized Mahalanobis

In [ ]:
D_euclidean = pairwise_distances(embeddings, metric="euclidean")
D_manhattan = pairwise_distances(embeddings, metric="manhattan")
D_cosine = pairwise_distances(embeddings, metric="cosine")

lw = LedoitWolf().fit(embeddings)
VI_lw = np.linalg.inv(lw.covariance_)
VI_lw = 0.5 * (VI_lw + VI_lw.T)
D_mahalanobis_lw = pairwise_distances(
    embeddings, metric="mahalanobis", VI=VI_lw
)

# Exact Euclidean representation of the fixed Mahalanobis quadratic form.
eigval, eigvec = np.linalg.eigh(VI_lw)
eigval = np.clip(eigval, 0.0, None)
W_maha = eigvec @ np.diag(np.sqrt(eigval))
X_maha = embeddings @ W_maha

D_maha_check = pairwise_distances(X_maha, metric="euclidean")
print("Ledoit-Wolf shrinkage:", lw.shrinkage_)
print("Maximum numerical difference in Mahalanobis check:",
      np.max(np.abs(D_mahalanobis_lw-D_maha_check)))


In [ ]:
distance_matrices = {
    "Euclidean": D_euclidean,
    "Manhattan": D_manhattan,
    "Cosine": D_cosine,
    "Mahalanobis_LW": D_mahalanobis_lw,
}
rows=[]
for name,D in distance_matrices.items():
    pred = AgglomerativeClustering(
        n_clusters=2, metric="precomputed", linkage="average"
    ).fit_predict(D)
    counts=np.bincount(pred)
    rows.append({
        "metric":name,
        "silhouette":silhouette_score(D,pred,metric="precomputed"),
        "cluster_sizes":"/".join(map(str,sorted(counts,reverse=True)))
    })
silhouette_summary=pd.DataFrame(rows)
display(silhouette_summary.round(4))
silhouette_summary.to_csv(OUTPUT_DIR/"distance_silhouette_unsupervised.csv",index=False)
assert set(silhouette_summary["cluster_sizes"]) == {"246/1"}

### Label-free kNN neighborhood stability (500 perturbations per level)

The original 500-replication design is retained. Runtime is reduced by using direct nearest-neighbor searches instead of recomputing complete pairwise matrices. The fixed Ledoit--Wolf Mahalanobis geometry is represented by an equivalent Euclidean linear transformation.

In [ ]:
rng=np.random.default_rng(SEED_MAIN)
feature_sd=embeddings.std(axis=0,ddof=1)
noise_levels=[0.01,0.05,0.10]
n_rep=500
k=PRIMARY_K

def fast_knn(X, metric, k):
    nn=NearestNeighbors(n_neighbors=k+1,metric=metric,n_jobs=-1).fit(X)
    return nn.kneighbors(X,return_distance=False)[:,1:]

original_neighbors={
    "Euclidean":fast_knn(embeddings,"euclidean",k),
    "Manhattan":fast_knn(embeddings,"manhattan",k),
    "Cosine":fast_knn(embeddings,"cosine",k),
    "Mahalanobis_LW":fast_knn(X_maha,"euclidean",k),
}

def preservation(old,new,k):
    return np.mean([
        len(set(old[i]).intersection(new[i]))/k
        for i in range(old.shape[0])
    ])

rows=[]
t0=time.time()
for noise_level in noise_levels:
    for rep in tqdm(range(n_rep),desc=f"Stability {noise_level:.0%}"):
        Xp=embeddings+rng.normal(
            0,noise_level*feature_sd,size=embeddings.shape
        )
        Xp_maha=Xp@W_maha
        new_neighbors={
            "Euclidean":fast_knn(Xp,"euclidean",k),
            "Manhattan":fast_knn(Xp,"manhattan",k),
            "Cosine":fast_knn(Xp,"cosine",k),
            "Mahalanobis_LW":fast_knn(Xp_maha,"euclidean",k),
        }
        for metric,new in new_neighbors.items():
            rows.append({
                "noise":noise_level,
                "rep":rep+1,
                "metric":metric,
                "mean_preservation":preservation(
                    original_neighbors[metric],new,k
                )
            })

stability_final=pd.DataFrame(rows)
stability_summary=(
    stability_final.groupby(["noise","metric"])["mean_preservation"]
    .agg(mean="mean",sd="std",median="median",
         q025=lambda x:x.quantile(.025),
         q975=lambda x:x.quantile(.975))
    .reset_index()
)
display(stability_summary.round(4))
print(f"Total stability runtime: {(time.time()-t0)/60:.1f} minutes")
stability_final.to_csv(OUTPUT_DIR/"knn_stability_500rep.csv",index=False)
stability_summary.to_csv(OUTPUT_DIR/"knn_stability_summary.csv",index=False)


## 4. Complete prompt blocks

In [ ]:
ps=(df.groupby("prompt_id")["label"].value_counts().unstack(fill_value=0))
complete_ids=ps.index[(ps["Human"]==1)&(ps["AI"]==3)].to_numpy()

df_complete=df[df["prompt_id"].isin(complete_ids)].copy()
df_complete["original_index"]=df_complete.index
df_complete=df_complete.reset_index(drop=True)
embeddings_complete=embeddings[df_complete["original_index"].to_numpy()]

print("Complete prompts:",len(complete_ids))
print(df_complete["label"].value_counts())
assert len(complete_ids)==61 and len(df_complete)==244

## 5. Final kNN graph

Union/OR symmetrization is used. Cosine **similarity** is edge strength for weighted clustering; cosine **distance** is path cost for weighted betweenness.

In [ ]:
def compute_topology_revision(X,d,k=10):
    temp=d.copy().reset_index(drop=True)
    labels=temp["label"].to_numpy()
    nn=NearestNeighbors(n_neighbors=k+1,metric="cosine").fit(X)
    distances,indices=nn.kneighbors(X)
    neigh=indices[:,1:]; ndist=distances[:,1:]

    temp["local_distance_mean"]=ndist.mean(axis=1)
    temp["local_density"]=1/temp["local_distance_mean"]
    temp["human_neighbor_ratio"]=[
        np.mean(labels[n]=="Human") for n in neigh
    ]

    G=nx.Graph()
    for i,row in temp.iterrows(): G.add_node(i,label=row["label"])
    for i in range(len(temp)):
        for j,dist in zip(neigh[i],ndist[i]):
            sim=1-dist
            if G.has_edge(i,j):
                if dist<G[i][j]["distance"]:
                    G[i][j]["distance"]=float(dist)
                    G[i][j]["similarity"]=float(sim)
            else:
                G.add_edge(i,j,distance=float(dist),similarity=float(sim))

    temp["graph_degree"]=temp.index.map(dict(G.degree()))
    temp["graph_clustering"]=temp.index.map(nx.clustering(G,weight="similarity"))
    temp["graph_betweenness"]=temp.index.map(
        nx.betweenness_centrality(G,weight="distance")
    )
    assort=nx.attribute_assortativity_coefficient(G,"label")
    return temp,assort

## 6. Primary k=10 observed results

In [ ]:
metrics=["local_density","local_distance_mean","graph_degree",
         "graph_clustering","graph_betweenness","human_neighbor_ratio"]

observed_complete,assort_observed=compute_topology_revision(
    embeddings_complete,df_complete,k=PRIMARY_K
)
means=observed_complete.groupby("label")[metrics].mean()
observed_diff=means.loc["AI"]-means.loc["Human"]
primary=pd.DataFrame({
    "Human_mean":means.loc["Human"],
    "AI_mean":means.loc["AI"],
    "AI_minus_Human":observed_diff
}).reset_index(names="metric")
display(primary.round(6))
print("Assortativity:",assort_observed)
primary.to_csv(OUTPUT_DIR/"primary_k10_observed.csv",index=False)

## Figure 1. Global UMAP projection — original manuscript specification

The exact UMAP settings used for the manuscript figure were recovered from the original working notebook: `n_neighbors=10`, `min_dist=0.1`, cosine metric, and `random_state=42`. The projection is descriptive only.


In [ ]:
# Original manuscript UMAP specification recovered from the working notebook.
# UMAP is descriptive only; inferential analyses use the original embedding space.
reducer = umap.UMAP(
    n_neighbors=10,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

umap_coords = reducer.fit_transform(embeddings)

df["UMAP1"] = umap_coords[:, 0]
df["UMAP2"] = umap_coords[:, 1]

plt.figure(figsize=(8,6))

for label in df["label"].unique():
    sub = df[df["label"] == label]
    plt.scatter(
        sub["UMAP1"], sub["UMAP2"],
        label=label, alpha=0.7
    )

plt.legend()
plt.title("UMAP - Human vs AI")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.savefig(
    OUTPUT_DIR/"UMAP_humanvsAI.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()


## Figure 2. kNN graph over UMAP space — original manuscript specification

This cell reproduces the original manuscript visualization on the full cleaned corpus. It is intentionally kept separate from the revised prompt-aware inferential graph analysis. The figure is descriptive only.


In [ ]:
# Original manuscript kNN visualization recovered from the working notebook.
# IMPORTANT: this graph is used for visualization only.
# Revised inferential graph statistics are computed separately above/below
# with the prompt-aware procedure and cosine distance as shortest-path cost.

k_plot = 10
nn_plot = NearestNeighbors(
    n_neighbors=k_plot+1,
    metric="cosine"
)
nn_plot.fit(embeddings)

distances_plot, indices_plot = nn_plot.kneighbors(embeddings)
neighbor_indices_plot = indices_plot[:, 1:]
neighbor_distances_plot = distances_plot[:, 1:]

G_visual = nx.Graph()

for i, row in df.iterrows():
    G_visual.add_node(
        i,
        sample_id=row["sample_id"],
        label=row["label"]
    )

for i in range(len(df)):
    for j, dist in zip(
        neighbor_indices_plot[i],
        neighbor_distances_plot[i]
    ):
        similarity = 1 - dist
        G_visual.add_edge(i, j, weight=similarity)

plt.figure(figsize=(10,8))

for edge in G_visual.edges():
    i, j = edge
    plt.plot(
        [df.loc[i, "UMAP1"], df.loc[j, "UMAP1"]],
        [df.loc[i, "UMAP2"], df.loc[j, "UMAP2"]],
        alpha=0.08,
        linewidth=0.5
    )

for label in df["label"].unique():
    sub = df[df["label"] == label]
    plt.scatter(
        sub["UMAP1"], sub["UMAP2"],
        label=label, alpha=0.8
    )

plt.legend()
plt.title("kNN Graph over UMAP space")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.savefig(
    OUTPUT_DIR/"kNN_graph_UMAP_space.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()


## 7. Prompt-block permutation inference

In [ ]:
def prepare_fixed_graph(X,d,k):
    nn=NearestNeighbors(n_neighbors=k+1,metric="cosine").fit(X)
    distances,indices=nn.kneighbors(X)
    neigh=indices[:,1:]; ndist=distances[:,1:]
    G=nx.Graph(); G.add_nodes_from(range(len(d)))
    for i in range(len(d)):
        for j,dist in zip(neigh[i],ndist[i]):
            sim=1-dist
            if G.has_edge(i,j):
                if dist<G[i][j]["distance"]:
                    G[i][j]["distance"]=float(dist); G[i][j]["similarity"]=float(sim)
            else:
                G.add_edge(i,j,distance=float(dist),similarity=float(sim))
    fixed=pd.DataFrame({
        "local_density":1/ndist.mean(axis=1),
        "local_distance_mean":ndist.mean(axis=1),
        "graph_degree":pd.Series(dict(G.degree())),
        "graph_clustering":pd.Series(nx.clustering(G,weight="similarity")),
        "graph_betweenness":pd.Series(nx.betweenness_centrality(G,weight="distance")),
    })
    return neigh,G,fixed

prompt_blocks=[
    np.flatnonzero(df_complete["prompt_id"].to_numpy()==pid)
    for pid in np.sort(df_complete["prompt_id"].unique())
]
assert all(len(b)==4 for b in prompt_blocks)

def block_permutation(k,n_perm,seed):
    topology,assort=compute_topology_revision(embeddings_complete,df_complete,k)
    gm=topology.groupby("label")[metrics].mean()
    obs=gm.loc["AI"]-gm.loc["Human"]
    neigh,G,fixed=prepare_fixed_graph(embeddings_complete,df_complete,k)

    rng=np.random.default_rng(seed); rows=[]
    for _ in tqdm(range(n_perm),desc=f"Block permutations k={k}"):
        lab=np.array(["AI"]*len(df_complete),dtype=object)
        for block in prompt_blocks: lab[rng.choice(block)]="Human"
        hm=lab=="Human"; am=lab=="AI"; row={}
        for m in ["local_density","local_distance_mean","graph_degree",
                  "graph_clustering","graph_betweenness"]:
            v=fixed[m].to_numpy()
            row[m]=v[am].mean()-v[hm].mean()
        hnr=np.array([np.mean(lab[n]=="Human") for n in neigh])
        row["human_neighbor_ratio"]=hnr[am].mean()-hnr[hm].mean()
        nx.set_node_attributes(G,{i:lab[i] for i in range(len(lab))},"label")
        row["assortativity"]=nx.attribute_assortativity_coefficient(G,"label")
        rows.append(row)

    null_df=pd.DataFrame(rows)
    observed={m:float(obs[m]) for m in metrics}
    observed["assortativity"]=float(assort)
    summary=[]
    for m,o in observed.items():
        null=null_df[m].dropna().to_numpy()
        mu=null.mean()
        extreme=np.sum(np.abs(null-mu)>=abs(o-mu))
        summary.append({"k":k,"metric":m,"observed":o,"null_mean":mu,
                        "p_perm":(extreme+1)/(len(null)+1)})
    return pd.DataFrame(summary),null_df

In [ ]:
# Main k=10: 10,000 permutations.
summary_k10,perm_k10=block_permutation(10,10000,SEED_MAIN)
display(summary_k10.round(6))
summary_k10.to_csv(OUTPUT_DIR/"prompt_block_permutation_summary_k10.csv",index=False)
perm_k10.to_csv(OUTPUT_DIR/"prompt_block_permutations_k10_10000.csv",index=False)

## 8. k sensitivity

In [ ]:
parts=[]
for k_test,seed in [(5,5005),(15,5015),(20,5020)]:
    s,null=block_permutation(k_test,5000,seed)
    parts.append(s)
    null.to_csv(OUTPUT_DIR/f"prompt_block_permutations_k{k_test}_5000.csv",index=False)

k_sensitivity_final=pd.concat(
    [parts[0],summary_k10,parts[1],parts[2]],ignore_index=True
).sort_values(["k","metric"]).reset_index(drop=True)

display(k_sensitivity_final.round(6))
k_sensitivity_final.to_csv(OUTPUT_DIR/"k_sensitivity_final.csv",index=False)

## 9. Prompt-balanced resampling — 1,000 graphs

In [ ]:
def balanced_resampling(n_rep=1000,k=10,seed=12345):
    rng=np.random.default_rng(seed)
    pids=np.sort(df_complete["prompt_id"].unique())
    humans=np.flatnonzero(df_complete["label"].to_numpy()=="Human")
    rows=[]
    for rep in tqdm(range(n_rep),desc="Prompt-balanced resampling"):
        selected=[]
        for pid in pids:
            ai=np.flatnonzero(
                (df_complete["prompt_id"].to_numpy()==pid)&
                (df_complete["label"].to_numpy()=="AI")
            )
            selected.append(rng.choice(ai))
        pos=np.concatenate([humans,np.asarray(selected,dtype=int)])
        d=df_complete.iloc[pos].copy().reset_index(drop=True)
        X=embeddings_complete[pos]
        topo,assort=compute_topology_revision(X,d,k)
        row={"replication":rep+1,"assortativity":assort}
        for m in metrics:
            h=topo.loc[topo["label"]=="Human",m].mean()
            a=topo.loc[topo["label"]=="AI",m].mean()
            row[f"{m}_human"]=h; row[f"{m}_ai"]=a; row[f"{m}_diff"]=a-h
        rows.append(row)
    return pd.DataFrame(rows)

balanced=balanced_resampling(1000,PRIMARY_K,SEED_BALANCED)
balanced.to_csv(OUTPUT_DIR/"prompt_balanced_k10_1000.csv",index=False)

rows=[]
for m in metrics:
    h=balanced[f"{m}_human"]; a=balanced[f"{m}_ai"]; d=balanced[f"{m}_diff"]
    rows.append({"metric":m,"Human_mean":h.mean(),"AI_mean":a.mean(),
                 "AI_minus_Human":d.mean(),"q025":np.percentile(d,2.5),
                 "q975":np.percentile(d,97.5),
                 "Pr_AI_gt_Human":np.mean(d>0),"Pr_AI_lt_Human":np.mean(d<0)})
balanced_summary=pd.DataFrame(rows)
display(balanced_summary.round(6))
print("Assortativity:",balanced["assortativity"].mean(),
      np.percentile(balanced["assortativity"],[2.5,97.5]))
balanced_summary.to_csv(OUTPUT_DIR/"prompt_balanced_k10_summary.csv",index=False)

## 10. Persistent homology — equal-size 64 vs 64

In [ ]:
human_X=embeddings[df["label"].to_numpy()=="Human"]
ai_X=embeddings[df["label"].to_numpy()=="AI"]

with warnings.catch_warnings():
    warnings.filterwarnings("ignore",message="The input point cloud has more columns than rows")
    human_ph=ripser(human_X,maxdim=2)

h1=human_ph["dgms"][1]
hp=h1[:,1]-h1[:,0]
hp=hp[np.isfinite(hp)]
human_values={"H1_loops":len(hp),"Mean_persistence":hp.mean(),"Max_persistence":hp.max()}
print(human_values)

rng=np.random.default_rng(SEED_PH); rows=[]
with warnings.catch_warnings():
    warnings.filterwarnings("ignore",message="The input point cloud has more columns than rows")
    for rep in tqdm(range(500),desc="Equal-n PH"):
        idx=rng.choice(len(ai_X),size=64,replace=False)
        ph=ripser(ai_X[idx],maxdim=2)["dgms"][1]
        p=ph[:,1]-ph[:,0]; p=p[np.isfinite(p)]
        rows.append({"rep":rep+1,"H1_loops":len(p),
                     "Mean_persistence":p.mean(),"Max_persistence":p.max()})
ph_equal=pd.DataFrame(rows)
ph_equal.to_csv(OUTPUT_DIR/"PH_equal_n_AI_500.csv",index=False)

rows=[]
for m,h in human_values.items():
    a=ph_equal[m].to_numpy()
    rows.append({"metric":m,"Human_value":h,"AI_mean_equal_n":a.mean(),
                 "AI_sd_equal_n":a.std(ddof=1),"AI_median_equal_n":np.median(a),
                 "AI_q025":np.percentile(a,2.5),"AI_q975":np.percentile(a,97.5),
                 "Pr_AI_gt_Human":np.mean(a>h),"Pr_AI_lt_Human":np.mean(a<h)})
ph_summary=pd.DataFrame(rows)
display(ph_summary.round(6))
ph_summary.to_csv(OUTPUT_DIR/"PH_equal_n_summary.csv",index=False)

## Figure 3. Persistence diagrams — original manuscript specification

The original manuscript figure displays persistent homology through `maxdim=2` (H0, H1, H2) for the full Human and AI point clouds. The revised quantitative robustness analysis remains the separate equal-size H1 resampling analysis.


In [ ]:
# Original manuscript persistence-diagram specification.
# The figure displays H0, H1 and H2 for the full Human and AI point clouds.
# The revised quantitative comparison reported in the manuscript focuses on H1
# and controls unequal sample size through the separate equal-n resampling analysis.

human_embeddings = embeddings[
    df["label"] == "Human"
]

ai_embeddings = embeddings[
    df["label"] == "AI"
]

human_ph = ripser(
    human_embeddings,
    maxdim=2
)

ai_ph = ripser(
    ai_embeddings,
    maxdim=2
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12,5)
)

plot_diagrams(
    human_ph["dgms"],
    show=False,
    ax=axes[0]
)
axes[0].set_title(
    "Human narratives",
    fontsize=13
)

plot_diagrams(
    ai_ph["dgms"],
    show=False,
    ax=axes[1]
)
axes[1].set_title(
    "AI-generated narratives",
    fontsize=13
)

plt.suptitle(
    "Persistent homology of semantic manifolds",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR/"persistence_diagrams_human_ai.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


## 11. Manuscript-level reproducibility checks

In [ ]:
expected={
    "local_density":0.9999,
    "local_distance_mean":-0.0771,
    "graph_degree":2.0765,
    "graph_clustering":0.1000,
    "graph_betweenness":0.0006,
    "human_neighbor_ratio":-0.2596,
}
for m,target in expected.items():
    assert np.isclose(float(observed_diff[m]),target,atol=5e-5),(m,observed_diff[m],target)
assert np.isclose(assort_observed,0.2421,atol=5e-5)

k20=float(k_sensitivity_final.query(
    "k==20 and metric=='human_neighbor_ratio'"
)["observed"].iloc[0])
assert np.isclose(k20,-0.2290,atol=5e-5),(k20,-0.2290)

print("All manuscript-level numerical checks passed.")
print("\nOutput files:")
for p in sorted(OUTPUT_DIR.glob("*")): print(" -",p.name)

## Interpretation boundary
Results generated by this notebook characterize the analyzed corpus and embedding representation. They do not establish universal topological properties of Human or AI-generated language. The AI narratives are attributable to the OpenAI `gpt-3.5-turbo-0301` model snapshot. However, the original prompt template and generation-parameter configuration (`temperature`, `top_p`, and maximum-token settings) were not retained and cannot be reconstructed reliably.

## Output manifest

This final cell lists the generated replication tables and manuscript figures.


In [ ]:
print("AI generation model:",GEN_MODEL)
print("Output directory:",OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -",p.name)
